In [1]:
import torch
import numpy as np
from PIL import Image
import torchvision.transforms as T
from sklearn.metrics.pairwise import cosine_similarity
import matplotlib.pyplot as plt
import os

In [2]:
model = torch.hub.load('facebookresearch/dinov2', 'dinov2_vits14')
model.eval()

Downloading: "https://github.com/facebookresearch/dinov2/zipball/main" to /home/joaodev/.cache/torch/hub/main.zip


/home/joaodev/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/home/joaodev/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/home/joaodev/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")


Downloading: "https://dl.fbaipublicfiles.com/dinov2/dinov2_vits14/dinov2_vits14_pretrain.pth" to /home/joaodev/.cache/torch/hub/checkpoints/dinov2_vits14_pretrain.pth


100.0%


DinoVisionTransformer(
  (patch_embed): PatchEmbed(
    (proj): Conv2d(3, 384, kernel_size=(14, 14), stride=(14, 14))
    (norm): Identity()
  )
  (blocks): ModuleList(
    (0-11): 12 x NestedTensorBlock(
      (norm1): LayerNorm((384,), eps=1e-06, elementwise_affine=True)
      (attn): MemEffAttention(
        (qkv): Linear(in_features=384, out_features=1152, bias=True)
        (proj): Linear(in_features=384, out_features=384, bias=True)
        (proj_drop): Dropout(p=0.0, inplace=False)
      )
      (ls1): LayerScale()
      (drop_path1): Identity()
      (norm2): LayerNorm((384,), eps=1e-06, elementwise_affine=True)
      (mlp): Mlp(
        (fc1): Linear(in_features=384, out_features=1536, bias=True)
        (act): GELU(approximate='none')
        (fc2): Linear(in_features=1536, out_features=384, bias=True)
        (drop): Dropout(p=0.0, inplace=False)
      )
      (ls2): LayerScale()
      (drop_path2): Identity()
    )
  )
  (norm): LayerNorm((384,), eps=1e-06, elementwise_affi

In [3]:
transform = T.Compose([
    T.Resize(256),
    T.CenterCrop(224),
    T.ToTensor(),
    T.Normalize(mean=(0.485, 0.456, 0.406),
                std=(0.229, 0.224, 0.225)),
])

In [4]:
def load_image(path):
    img = Image.open(path).convert("RGB")
    return transform(img).unsqueeze(0), img

In [5]:
def get_embedding(image_tensor):
    with torch.no_grad():
        emb = model(image_tensor)
    emb = emb.squeeze().numpy()
    emb = emb / np.linalg.norm(emb)  # normalização
    return emb

In [ ]:
IMAGE_FOLDER = "./data/cropped_cards"

embeddings_db = []
paths = []
images = []

for file in os.listdir(IMAGE_FOLDER):
    path = os.path.join(IMAGE_FOLDER, file)
    try:
        tensor, img = load_image(path)
        emb = get_embedding(tensor)
        embeddings_db.append(emb)
        paths.append(path)
        images.append(img)
    except:
        print(f"Erro ao processar {path}")

embeddings_db = np.array(embeddings_db)

print(f"Total de cartas processadas: {len(paths)}")

FileNotFoundError: [Errno 2] No such file or directory: './cartas'